In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1Y91cC6epwsrlw_fxyi3zC3GnM9qGZ_nG", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why Striped Attention
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_why_striped_attention.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Striped Attention for Causal LM -- Vizuara

## 1. Why Does This Matter?

In the previous notebook, we implemented Ring Attention and saw a critical problem: under **causal masking** with contiguous chunks, GPUs holding early tokens do far less work than GPUs holding late tokens. With 4 GPUs, utilization drops to ~62.5%. With 16 GPUs, it falls to ~53%.

**Striped Attention** (Brandon et al., 2023) solves this with a simple idea: instead of assigning contiguous blocks of tokens to each GPU, assign tokens in a **round-robin** (striped) pattern. GPU 0 gets tokens [0, 4, 8, ...], GPU 1 gets [1, 5, 9, ...], etc.

In this notebook, we will:
1. **Implement** the striped token assignment
2. **Compute** the causal mask under striped assignment and verify load balance
3. **Build** full Striped Ring Attention and verify correctness
4. **Compare** utilization between contiguous and striped approaches
5. **Analyze** how Striped Attention scales to many GPUs

By the end, you will understand why Meta uses Striped Attention for Llama 3.1's 128K-token training.

In [ ]:
#@title 🎧 Transition: Section2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_section2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Contiguous vs Striped Assignment

Let us start by visualizing the difference between contiguous and striped token assignment.

In [ ]:
#@title 🎧 Code Walkthrough: Visualize Assignment Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_visualize_assignment_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
#@title 🎧 What to Look For: Visualize Assignment Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_visualize_assignment_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visualize_assignment(S=16, N=4):
    """Visualize contiguous vs striped token assignment."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5))
    colors = plt.cm.Set2(np.linspace(0, 1, N))

    # Contiguous assignment
    chunk_size = S // N
    for pos in range(S):
        gpu = pos // chunk_size
        ax1.barh(0, 1, left=pos, color=colors[gpu], edgecolor='black', linewidth=0.5)
        ax1.text(pos + 0.5, 0, str(pos), ha='center', va='center', fontsize=8)

    ax1.set_title('Contiguous Assignment', fontsize=13, fontweight='bold')
    ax1.set_xlim(0, S)
    ax1.set_ylim(-0.5, 0.5)
    ax1.set_yticks([])
    ax1.set_xlabel('Token Position', fontsize=11)

    # Striped assignment
    for pos in range(S):
        gpu = pos % N
        ax2.barh(0, 1, left=pos, color=colors[gpu], edgecolor='black', linewidth=0.5)
        ax2.text(pos + 0.5, 0, str(pos), ha='center', va='center', fontsize=8)

    ax2.set_title('Striped (Round-Robin) Assignment', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, S)
    ax2.set_ylim(-0.5, 0.5)
    ax2.set_yticks([])
    ax2.set_xlabel('Token Position', fontsize=11)

    # Legend
    patches = [mpatches.Patch(facecolor=colors[i], edgecolor='black',
                              label=f'GPU {i}') for i in range(N)]
    fig.legend(handles=patches, loc='upper right', fontsize=9, ncol=N)

    plt.tight_layout()
    plt.show()

    # Print assignment tables
    print("\nContiguous Assignment:")
    for g in range(N):
        tokens = list(range(g * chunk_size, (g + 1) * chunk_size))
        print(f"  GPU {g}: tokens {tokens}")

    print("\nStriped Assignment:")
    for g in range(N):
        tokens = list(range(g, S, N))
        print(f"  GPU {g}: tokens {tokens}")

    print(f"\nKey difference: In striped assignment, every GPU has a MIX")
    print(f"of early and late tokens. No GPU is stuck with only early tokens.")


visualize_assignment(S=16, N=4)

In [ ]:
#@title 🎧 Transition: Section3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_section3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Causal Mask Under Striped Assignment

Under causal masking, token at position $p$ can only attend to positions $\leq p$. Let us compute the number of non-masked entries for each GPU pair under both contiguous and striped assignment.

In [ ]:
#@title 🎧 Code Walkthrough: Compute Causal Work Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_compute_causal_work_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_causal_work(S, N, assignment='contiguous'):
    """
    Compute per-GPU work (non-masked logits) under causal masking.

    Returns:
        work_matrix: (N, N) array where work_matrix[i][j] = number of non-masked
                     logits when GPU i's queries attend to GPU j's keys
        total_work: (N,) total work per GPU
    """
    chunk_size = S // N

    if assignment == 'contiguous':
        # GPU g holds tokens [g*chunk_size, (g+1)*chunk_size)
        gpu_tokens = {g: list(range(g * chunk_size, (g + 1) * chunk_size)) for g in range(N)}
    elif assignment == 'striped':
        # GPU g holds tokens [g, g+N, g+2N, ...]
        gpu_tokens = {g: list(range(g, S, N)) for g in range(N)}
    else:
        raise ValueError(f"Unknown assignment: {assignment}")

    work_matrix = np.zeros((N, N), dtype=int)

    for qi_gpu in range(N):
        for kv_gpu in range(N):
            q_positions = gpu_tokens[qi_gpu]
            kv_positions = gpu_tokens[kv_gpu]

            count = 0
            for q_pos in q_positions:
                for k_pos in kv_positions:
                    if k_pos <= q_pos:  # causal: can attend to past
                        count += 1
            work_matrix[qi_gpu, kv_gpu] = count

    total_work = work_matrix.sum(axis=1)  # total across all KV chunks
    return work_matrix, total_work


In [ ]:
#@title 🎧 What to Look For: Compute Causal Work Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_compute_causal_work_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
S, N = 16, 4

# Contiguous
wm_cont, tw_cont = compute_causal_work(S, N, 'contiguous')
print("CONTIGUOUS: Work matrix (rows=Q GPU, cols=KV GPU)")
print("Each entry = number of non-masked Q-K pairs\n")
header = "      " + "".join(f"KV{j:>4}" for j in range(N))
print(header)
for i in range(N):
    row = f"Q GPU{i}" + "".join(f"{wm_cont[i,j]:>5}" for j in range(N))
    print(row + f"  | Total: {tw_cont[i]}")

max_cont = tw_cont.max()
util_cont = tw_cont.mean() / max_cont * 100
print(f"\nUtilization: {util_cont:.1f}%")

print("\n" + "=" * 60)

# Striped
wm_strip, tw_strip = compute_causal_work(S, N, 'striped')
print("\nSTRIPED: Work matrix (rows=Q GPU, cols=KV GPU)\n")
print(header)
for i in range(N):
    row = f"Q GPU{i}" + "".join(f"{wm_strip[i,j]:>5}" for j in range(N))
    print(row + f"  | Total: {tw_strip[i]}")

max_strip = tw_strip.max()
util_strip = tw_strip.mean() / max_strip * 100
print(f"\nUtilization: {util_strip:.1f}%")

print(f"\nImprovement: {util_cont:.1f}% -> {util_strip:.1f}%")

In [ ]:
#@title 🎧 What to Look For: Visualize Work Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_visualize_work_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the work distribution comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Contiguous
colors_bar = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
bars1 = ax1.bar(range(N), tw_cont / max_cont * 100, color=colors_bar,
                edgecolor='black', linewidth=1.2)
ax1.set_xlabel('GPU', fontsize=12)
ax1.set_ylabel('Relative Work (%)', fontsize=12)
ax1.set_title(f'Contiguous Chunks\nUtilization: {util_cont:.1f}%',
              fontsize=13, fontweight='bold', color='red')
ax1.set_xticks(range(N))
ax1.set_xticklabels([f'GPU {i}' for i in range(N)])
ax1.set_ylim(0, 120)
for bar, w in zip(bars1, tw_cont):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             f'{w/max_cont*100:.0f}%', ha='center', fontsize=11, fontweight='bold')

# Striped
bars2 = ax2.bar(range(N), tw_strip / max_strip * 100, color=colors_bar,
                edgecolor='black', linewidth=1.2)
ax2.set_xlabel('GPU', fontsize=12)
ax2.set_ylabel('Relative Work (%)', fontsize=12)
ax2.set_title(f'Striped (Round-Robin)\nUtilization: {util_strip:.1f}%',
              fontsize=13, fontweight='bold', color='green')
ax2.set_xticks(range(N))
ax2.set_xticklabels([f'GPU {i}' for i in range(N)])
ax2.set_ylim(0, 120)
for bar, w in zip(bars2, tw_strip):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             f'{w/max_strip*100:.0f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Causal Attention Load Balance: Contiguous vs Striped',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Striped assignment nearly equalizes the workload across all GPUs.")
print("The small remaining imbalance comes from GPU 0 having slightly fewer")
print("future tokens to be attended to than GPU N-1.")

In [ ]:
#@title 🎧 Transition: Section4 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_section4_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Implementing Striped Ring Attention

Now let us implement the full Striped Ring Attention algorithm. The key changes from standard Ring Attention:
1. **Token assignment**: Tokens are assigned round-robin instead of contiguously
2. **Causal mask**: Must track original positions to apply the causal constraint correctly
3. **Output reassembly**: Results must be un-striped back to original order

In [ ]:
#@title 🎧 Code Walkthrough: Stripe Unstripe Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_stripe_unstripe_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def stripe_tensor(x, num_gpus):
    """
    Rearrange a tensor from contiguous to striped layout.
    Input x: (B, S, d_h) in original order
    Returns: list of N tensors, each (B, S//N, d_h),
             where GPU g gets positions [g, g+N, g+2N, ...]
    """
    B, S, d_h = x.shape
    chunks = []
    for g in range(num_gpus):
        indices = list(range(g, S, num_gpus))
        chunks.append(x[:, indices, :])
    return chunks


def unstripe_tensor(chunks, num_gpus):
    """
    Rearrange from striped layout back to original order.
    Input: list of N tensors, each (B, S//N, d_h)
    Returns: (B, S, d_h) in original token order
    """
    B = chunks[0].shape[0]
    d_h = chunks[0].shape[-1]
    chunk_size = chunks[0].shape[1]
    S = chunk_size * num_gpus

    output = torch.zeros(B, S, d_h, dtype=chunks[0].dtype)
    for g in range(num_gpus):
        indices = list(range(g, S, num_gpus))
        output[:, indices, :] = chunks[g]
    return output


# Test stripe/unstripe roundtrip
B, S, d_h, N = 1, 16, 4, 4
x = torch.arange(S).float().unsqueeze(0).unsqueeze(-1).expand(B, S, d_h)


In [ ]:
#@title 🎧 What to Look For: Stripe Unstripe Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_stripe_unstripe_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
chunks = stripe_tensor(x, N)
for g in range(N):
    positions = chunks[g][0, :, 0].tolist()
    print(f"GPU {g} gets positions: {[int(p) for p in positions]}")

reconstructed = unstripe_tensor(chunks, N)
match = torch.allclose(x, reconstructed)
print(f"\nRoundtrip match: {'YES' if match else 'NO'}")

In [ ]:
#@title 🎧 Code Walkthrough: Striped Ring Attention Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_striped_ring_attention_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def striped_ring_attention_causal(Q_full, K_full, V_full, num_gpus=4, verbose=True):
    """
    Striped Ring Attention with causal masking.

    Q_full, K_full, V_full: (B, S, d_h) in original token order
    Returns: (B, S, d_h) in original token order
    """
    B, S, d_h = Q_full.shape
    N = num_gpus
    chunk_size = S // N

    # Step 1: Stripe the tokens
    Q_chunks = stripe_tensor(Q_full, N)
    K_chunks = stripe_tensor(K_full, N)
    V_chunks = stripe_tensor(V_full, N)

    # Track original positions for causal masking
    # GPU g holds positions [g, g+N, g+2N, ...]
    gpu_positions = {g: list(range(g, S, N)) for g in range(N)}

    # Initialize accumulators for each GPU
    running_max = [torch.full((B, chunk_size, 1), float('-inf')) for _ in range(N)]
    running_sum = [torch.zeros((B, chunk_size, 1)) for _ in range(N)]
    running_out = [torch.zeros((B, chunk_size, d_h)) for _ in range(N)]

    # Current KV on each GPU
    K_current = [K_chunks[g].clone() for g in range(N)]
    V_current = [V_chunks[g].clone() for g in range(N)]
    kv_source = list(range(N))  # Which GPU's KV each GPU currently holds

    work_per_gpu = [0] * N

    for step in range(N):
        if verbose:
            kv_map = [f"GPU{g}:KV{kv_source[g]}" for g in range(N)]
            print(f"Step {step+1}: {' | '.join(kv_map)}")

        for g in range(N):
            # GPU g has queries at positions gpu_positions[g]
            # Current KV comes from GPU kv_source[g] at positions gpu_positions[kv_source[g]]
            q_pos = gpu_positions[g]
            kv_pos = gpu_positions[kv_source[g]]

            # Compute logits
            logits = Q_chunks[g] @ K_current[g].transpose(-2, -1) / (d_h ** 0.5)

            # Apply causal mask based on ORIGINAL positions
            for qi in range(chunk_size):
                for ki in range(chunk_size):
                    if kv_pos[ki] > q_pos[qi]:  # KV position is in the future
                        logits[:, qi, ki] = float('-inf')
                    else:
                        work_per_gpu[g] += 1

            # Skip if all masked
            if logits.max() == float('-inf'):
                continue

            # Online softmax update
            chunk_max = logits.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(running_max[g], chunk_max)

            alpha = torch.exp(running_max[g] - new_max)
            beta = torch.exp(logits - new_max)
            beta = torch.where(torch.isinf(logits) & (logits < 0),
                               torch.zeros_like(beta), beta)

            running_sum[g] = alpha * running_sum[g] + beta.sum(dim=-1, keepdim=True)
            running_out[g] = alpha * running_out[g] + beta @ V_current[g]
            running_max[g] = new_max

        # Rotate KV
        if step < N - 1:
            kv_buffer = [(K_current[g].clone(), V_current[g].clone(), kv_source[g])
                         for g in range(N)]
            for g in range(N):
                sender = (g - 1) % N
                K_current[g] = kv_buffer[sender][0]
                V_current[g] = kv_buffer[sender][1]
                kv_source[g] = kv_buffer[sender][2]

    # Normalize and un-stripe
    output_chunks = []
    for g in range(N):
        out = running_out[g] / running_sum[g]
        output_chunks.append(out)

    output = unstripe_tensor(output_chunks, N)
    return output, work_per_gpu


# Reference: standard causal attention
def standard_causal_attention(Q, K, V):
    B, S, d_h = Q.shape
    scores = Q @ K.transpose(-2, -1) / (d_h ** 0.5)
    # Apply causal mask
    mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
    scores.masked_fill_(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V


# Test
B, S, d_h = 1, 32, 8
torch.manual_seed(42)
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)


In [ ]:
#@title 🎧 Code Walkthrough: Striped Ring Attention Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_striped_ring_attention_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
ref = standard_causal_attention(Q, K, V)
out_striped, work_striped = striped_ring_attention_causal(Q, K, V, num_gpus=4, verbose=True)

diff = (ref - out_striped).abs().max().item()
print(f"\nMax difference from standard causal attention: {diff:.2e}")
print(f"Match: {'YES -- Striped Ring Attention is exact!' if diff < 1e-4 else 'NO -- bug'}")

In [ ]:
#@title 🎧 Transition: Section5 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_section5_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Load Balance Comparison: Contiguous vs Striped

Let us directly compare the workload distribution under both approaches.

In [ ]:
#@title 🎧 Code Walkthrough: Compare Load Balance Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_compare_load_balance_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compare_load_balance(S_values, N_values):
    """Compare utilization between contiguous and striped for various configs."""
    results = []

    for S in S_values:
        for N in N_values:
            if S % N != 0:
                continue

            _, tw_cont = compute_causal_work(S, N, 'contiguous')
            _, tw_strip = compute_causal_work(S, N, 'striped')

            util_cont = tw_cont.mean() / tw_cont.max() * 100
            util_strip = tw_strip.mean() / tw_strip.max() * 100

            results.append({
                'S': S, 'N': N,
                'util_cont': util_cont,
                'util_strip': util_strip,
                'improvement': util_strip - util_cont
            })

    return results


In [ ]:
#@title 🎧 What to Look For: Compare Load Balance Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_compare_load_balance_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
S_values = [16, 32, 64, 128]
N_values = [2, 4, 8]

results = compare_load_balance(S_values, N_values)

print(f"{'S':>6} | {'N':>3} | {'Contiguous':>12} | {'Striped':>10} | {'Improvement':>12}")
print("-" * 55)

for r in results:
    print(f"{r['S']:>6} | {r['N']:>3} | {r['util_cont']:>11.1f}% | {r['util_strip']:>9.1f}% | {r['improvement']:>+11.1f}%")

print("\nStriped consistently achieves near-100% utilization.")
print("The improvement grows as N increases (more GPUs = worse contiguous imbalance).")

In [ ]:
#@title 🎧 What to Look For: Visualize Utilization Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_visualize_utilization_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize utilization across GPU counts
fig, ax = plt.subplots(figsize=(10, 6))

# Theoretical contiguous: (N+1)/(2N)
N_range = np.arange(2, 33)
util_cont_theory = (N_range + 1) / (2 * N_range) * 100

ax.plot(N_range, util_cont_theory, 'r-o', markersize=5, linewidth=2,
        label='Contiguous (theory: (N+1)/2N)')

# Measured striped utilization (from compute_causal_work)
S_test = 128  # Use S=128 for all tests
N_test = [2, 4, 8, 16, 32]
util_strip_measured = []

for N in N_test:
    if S_test % N == 0:
        _, tw = compute_causal_work(S_test, N, 'striped')
        util_strip_measured.append(tw.mean() / tw.max() * 100)
    else:
        util_strip_measured.append(None)

valid_N = [n for n, u in zip(N_test, util_strip_measured) if u is not None]
valid_util = [u for u in util_strip_measured if u is not None]
ax.plot(valid_N, valid_util, 'g-s', markersize=8, linewidth=2,
        label=f'Striped (measured, S={S_test})')

ax.axhline(y=100, color='black', linestyle=':', alpha=0.5)
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5)
ax.text(30, 101, 'Ideal: 100%', fontsize=9, color='black')
ax.text(30, 51, '50% floor', fontsize=9, color='gray')

ax.set_xlabel('Number of GPUs (N)', fontsize=13)
ax.set_ylabel('Utilization (%)', fontsize=13)
ax.set_title('Causal Attention Utilization: Contiguous vs Striped',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(40, 105)

plt.tight_layout()
plt.show()

print("Contiguous utilization drops toward 50% as N grows.")
print("Striped utilization stays near 100% regardless of N.")
print("This is why Llama 3.1 uses Striped Attention for 128K context.")

In [ ]:
#@title 🎧 Transition: Section6 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_section6_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Detailed Causal Mask Heatmaps

Let us visualize the actual causal mask patterns for contiguous vs striped at the individual token level.

In [ ]:
#@title 🎧 Code Walkthrough: Visualize Causal Masks Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_visualize_causal_masks_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
#@title 🎧 What to Look For: Visualize Causal Masks Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_visualize_causal_masks_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visualize_causal_masks(S=16, N=4):
    """Visualize full causal mask colored by GPU pair for both assignments."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = plt.cm.Set2(np.linspace(0, 1, N))

    for ax, assignment, title in [(ax1, 'contiguous', 'Contiguous'),
                                   (ax2, 'striped', 'Striped')]:
        if assignment == 'contiguous':
            chunk_size = S // N
            pos_to_gpu = {p: p // chunk_size for p in range(S)}
        else:
            pos_to_gpu = {p: p % N for p in range(S)}

        # Build colored mask image (S x S x 3 RGB)
        img = np.ones((S, S, 3))  # white background

        for q_pos in range(S):
            for k_pos in range(S):
                q_gpu = pos_to_gpu[q_pos]
                k_gpu = pos_to_gpu[k_pos]

                if k_pos <= q_pos:  # causal: can attend
                    # Color by query GPU
                    img[q_pos, k_pos] = colors[q_gpu][:3]
                else:
                    # Masked: light gray
                    img[q_pos, k_pos] = [0.9, 0.9, 0.9]

        ax.imshow(img, origin='upper', aspect='equal')

        # Draw GPU boundary lines
        for g in range(1, N):
            if assignment == 'contiguous':
                pos = g * chunk_size - 0.5
                ax.axhline(y=pos, color='black', linewidth=1, alpha=0.5)
                ax.axvline(x=pos, color='black', linewidth=1, alpha=0.5)

        ax.set_xlabel('Key Position', fontsize=11)
        ax.set_ylabel('Query Position', fontsize=11)
        ax.set_title(f'{title} Assignment\n(colored by query GPU)', fontsize=13, fontweight='bold')

        if S <= 20:
            ax.set_xticks(range(S))
            ax.set_yticks(range(S))
            ax.set_xticklabels(range(S), fontsize=7)
            ax.set_yticklabels(range(S), fontsize=7)

    # Legend
    patches = [mpatches.Patch(facecolor=colors[i][:3], edgecolor='black',
                              label=f'GPU {i}') for i in range(N)]
    patches.append(mpatches.Patch(facecolor=[0.9, 0.9, 0.9], edgecolor='black',
                                   label='Masked'))
    fig.legend(handles=patches, loc='lower center', fontsize=9, ncol=N+1)

    plt.suptitle('Causal Mask: Token-Level View', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    print("Left (Contiguous): GPU 0's colored region is small (few non-masked entries).")
    print("Right (Striped): All GPUs' colored regions are roughly equal in area.")


visualize_causal_masks(S=16, N=4)

In [ ]:
#@title 🎧 Before You Start: Todo Exercise1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_todo_exercise1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Your Turn

### Exercise 1: Verify Striped Ring Attention at Scale

Test Striped Ring Attention with larger sequences and more GPU counts. Verify correctness by comparing to standard causal attention.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_todo_exercise1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Test Striped Ring Attention across configurations

configs = [
    (1, 64, 16, 2),    # B, S, d_h, N
    (1, 64, 16, 4),
    (1, 128, 32, 4),
    (1, 128, 32, 8),
    (1, 256, 64, 4),
    (1, 256, 64, 8),
]

print(f"{'Config':>30} | {'Max Diff':>12} | {'Status':>8}")
print("-" * 60)

for B, S, d_h, N in configs:
    torch.manual_seed(42)
    Q = torch.randn(B, S, d_h)
    K = torch.randn(B, S, d_h)
    V = torch.randn(B, S, d_h)

    # TODO: Compute reference with standard_causal_attention
    ref = None  # REPLACE THIS LINE

    # TODO: Compute striped ring attention output
    out = None  # REPLACE THIS LINE

    if ref is not None and out is not None:
        diff = (ref - out).abs().max().item()
        status = "PASS" if diff < 1e-3 else "FAIL"
    else:
        diff = float('nan')
        status = "TODO"

    config_str = f"B={B}, S={S}, d_h={d_h}, N={N}"
    print(f"{config_str:>30} | {diff:>12.2e} | {status:>8}")

print("\nAll should PASS. Note that differences may be slightly larger")
print("than non-causal Ring Attention due to the masked entries.")

In [ ]:
#@title 🎧 Before You Start: Todo Exercise2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_todo_exercise2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Compute Utilization Improvement for Large N

For $N = 2, 4, 8, 16, 32, 64$ GPUs, compute:
1. Contiguous utilization (formula: $(N+1)/(2N)$)
2. The percentage of wasted GPU time under contiguous assignment
3. The dollar cost saved by Striped Attention if you are renting H100 GPUs at \$3/hr each

In [ ]:
#@title 🎧 Before You Start: Todo Exercise2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_todo_exercise2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Compute cost savings from Striped Attention

cost_per_gpu_per_hour = 3.0  # $/hr for H100

N_values = [2, 4, 8, 16, 32, 64]

print(f"{'N GPUs':>7} | {'Contiguous Util':>16} | {'Wasted Time':>12} | {'Wasted $/hr':>12} | {'Saved by Striped':>17}")
print("-" * 75)

for N in N_values:
    # TODO: Compute contiguous utilization
    util_cont = None  # REPLACE: (N+1) / (2*N)

    # TODO: Compute wasted fraction
    wasted_frac = None  # REPLACE: 1 - util_cont

    # TODO: Compute cost wasted per hour (N GPUs * cost * wasted_fraction)
    cost_wasted = None  # REPLACE

    # TODO: Striped utilization is ~100%, so saved = cost_wasted
    saved = None  # REPLACE

    if all(x is not None for x in [util_cont, wasted_frac, cost_wasted, saved]):
        print(f"{N:>7} | {util_cont*100:>15.1f}% | {wasted_frac*100:>11.1f}% | ${cost_wasted:>10.2f} | ${saved:>15.2f}/hr")
    else:
        print(f"{N:>7} | {'TODO':>16} | {'TODO':>12} | {'TODO':>12} | {'TODO':>17}")

print("\nAt 64 GPUs, contiguous wastes ~$94/hr. Over a 30-day training run:")
print("  Wasted cost = $94 * 24 * 30 = $67,680")
print("  Striped Attention saves all of this.")

In [ ]:
#@title 🎧 Before You Start: Todo Exercise3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_todo_exercise3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: Implement a Striped Index Mapper

Write a function that, given a global token position and the number of GPUs, returns which GPU the token is on and its local index within that GPU's chunk.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_todo_exercise3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Implement the striped index mapper

def striped_index(global_pos, num_gpus):
    """
    Given a global token position and number of GPUs (striped assignment),
    return (gpu_id, local_index).

    Example with N=4:
      global_pos=0  -> GPU 0, local_idx=0
      global_pos=1  -> GPU 1, local_idx=0
      global_pos=4  -> GPU 0, local_idx=1
      global_pos=7  -> GPU 3, local_idx=1
    """
    # TODO: Compute gpu_id and local_index
    # Hint: gpu_id = global_pos % num_gpus
    #        local_index = global_pos // num_gpus

    gpu_id = None     # REPLACE
    local_index = None  # REPLACE
    return gpu_id, local_index


# Test
N = 4
S = 16
print(f"Striped assignment with N={N} GPUs, S={S} tokens:\n")
print(f"{'Global Pos':>10} | {'GPU':>4} | {'Local Index':>11}")
print("-" * 30)

for pos in range(S):
    gpu, local = striped_index(pos, N)
    if gpu is not None:
        print(f"{pos:>10} | {gpu:>4} | {local:>11}")
    else:
        print(f"{pos:>10} | {'TODO':>4} | {'TODO':>11}")

print("\nVerify: GPU 0 should have positions [0, 4, 8, 12]")
print("        GPU 3 should have positions [3, 7, 11, 15]")

In [ ]:
#@title 🎧 Transition: Section8 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_section8_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Putting It All Together: The Full Picture

Let us summarize the entire context parallelism pipeline from sequence to output.

In [ ]:
#@title 🎧 Listen: Full Pipeline Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_full_pipeline_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Summary visualization: the full CP pipeline
print("=" * 70)
print("CONTEXT PARALLELISM: THE FULL PIPELINE")
print("=" * 70)
print()
print("Input: Long sequence of S tokens")
print("       (e.g., S = 131,072 for a 128K context)")
print()
print("Step 1: PARTITION (Striped Assignment)")
print("  Token 0 -> GPU 0, Token 1 -> GPU 1, ..., Token N-1 -> GPU N-1")
print("  Token N -> GPU 0, Token N+1 -> GPU 1, ... (round-robin)")
print()
print("Step 2: COMPUTE Q, K, V locally")
print("  Each GPU computes Q_i, K_i, V_i for its local tokens")
print()
print("Step 3: RING ATTENTION (N steps)")
print("  For step t = 0, 1, ..., N-1:")
print("    a) Compute partial attention: Q_i x KV_current")
print("    b) Update online softmax accumulators")
print("    c) Send KV_current to next GPU, receive from previous")
print("    (Communication overlapped with computation)")
print()
print("Step 4: NORMALIZE")
print("  Each GPU: O_i = running_out / running_sum")
print()
print("Step 5: UN-STRIPE")
print("  Reassemble outputs in original token order")
print()
print("Memory per GPU: O(S^2 / N^2) for attention scores")
print("                O(S / N) with FlashAttention")
print()
print("Utilization:")
print("  Contiguous chunks: ~(N+1)/(2N) under causal masking")
print("  Striped chunks:    ~100% under causal masking")

In [ ]:
#@title 🎧 Listen: Section9 Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_30_section9_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary

In this notebook we implemented Striped Attention and showed why it is essential for efficient causal language model training:

1. **Contiguous chunking fails under causal masking**: Early-token GPUs are idle while late-token GPUs do all the work. Utilization drops to ~50% at scale.

2. **Striped (round-robin) assignment**: Each GPU gets a mix of early and late tokens. The causal mask creates roughly equal work across all GPUs.

3. **Numerically exact**: Striped Ring Attention produces the exact same output as standard causal attention, verified empirically.

4. **Practical impact**: At 64 GPUs, contiguous wastes ~49% of compute. Over a 30-day training run on H100s, that is tens of thousands of dollars saved.

5. **Production usage**: Meta's Llama 3.1 combines Striped Attention (for context parallelism) with Tensor Parallelism, Pipeline Parallelism, and Data Parallelism across hundreds of GPUs.

Context parallelism splits the sequence for attention. The next dimension to explore is **Pipeline Parallelism** -- splitting the model's layers across GPUs to form an assembly line.

In [ ]:
#@title 🎧 Listen: Final Summary Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_31_final_summary_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("SUMMARY: Striped Attention for Causal LM")
print("=" * 60)
print()
print("Key results:")
print("  - Contiguous chunks + causal mask = severe load imbalance")
print("  - Utilization: (N+1)/(2N) -> approaches 50% at large N")
print("  - Striped assignment: round-robin tokens across GPUs")
print("  - Every GPU gets a balanced mix of early and late tokens")
print("  - Utilization: ~100% regardless of N")
print("  - Used in Llama 3.1 for 128K-token training")
print()
print("The full context parallelism stack:")
print("  Ring Attention (communication) + Striped Assignment (load balance)")
print("  + FlashAttention (per-chunk memory) + Online Softmax (exact results)")